# 20 - v03c map.nc subset (runs on EDITO Datalab)

**Run this notebook inside EDITO JupyterLab**, not locally. It reads the 57 GB of partitioned map.nc files from the bucket via the fast intranet, selects a handful of time slices plus the key variables, and writes a compact subset (~200-400 MB) back to `DFM_OUTPUT_SUBSET/` for download to the home workstation.

**Why not run locally:** downloading 57 GB over a home link takes hours; most of it is variables (turbulence, MPI ghosts, diagnostics) we do not need for the v03c evaluation.

Steps:
1. Copy the 4 partition map.nc files from S3 to scratch (`mc cp`, fast on EDITO).
2. Merge + subset with `dfm_tools.open_partitioned_dataset`.
3. Write subset as a single NetCDF.
4. Upload subset back to `DFM_OUTPUT_SUBSET/`.

After this, run locally: `python scripts/edito_sync.py download-subset --model-dir model/dflowfm_v03c`.

## 1. Environment + paths (EDITO side)

In [ ]:
import os, subprocess, time
from pathlib import Path

# EDITO Jupyter: local scratch lives under /home/onyxia/work (adjust if yours differs).
SCRATCH = Path('/home/onyxia/work/v03c_map_local')
SCRATCH.mkdir(parents=True, exist_ok=True)

# S3 source and destination (mc alias 'edito' must be configured)
S3_SRC = 'edito/oidc-cmartinsjr/DFM_OUTPUT/'
S3_DST = 'edito/oidc-cmartinsjr/DFM_OUTPUT_SUBSET/'
SUBSET_FILE = SCRATCH / 'v03c_map_subset.nc'

# Sanity: mc must be on PATH on EDITO JupyterLab by default
mc_ok = subprocess.run(['mc', '--version'], capture_output=True, text=True)
print('mc:', mc_ok.stdout.splitlines()[0] if mc_ok.returncode == 0 else 'NOT FOUND')

# Try importing dfm_tools; install if missing
try:
    import dfm_tools as dfmt  # noqa: F401
    print(f'dfm_tools {dfmt.__version__}')
except ImportError:
    print('Installing dfm_tools...')
    subprocess.check_call(['pip', 'install', '--quiet', 'dfm_tools'])
    import dfm_tools as dfmt
    print(f'dfm_tools {dfmt.__version__}')

## 2. Pull partitioned maps from S3 to scratch

14 GB x 4 partitions = 56 GB. On EDITO intranet this is a few minutes; on a home link would be hours.
If scratch already has the files, skip the copy.

In [ ]:
t0 = time.time()
for part in range(4):
    fname = f'Stagnone_dxy01_15m_{part:04d}_map.nc'
    local = SCRATCH / fname
    if local.exists() and local.stat().st_size > 1_000_000_000:
        print(f'  [skip] {fname} already local ({local.stat().st_size/1e9:.1f} GB)')
        continue
    t1 = time.time()
    subprocess.run(['mc', 'cp', f'{S3_SRC}{fname}', str(local)], check=True)
    print(f'  [ok]   {fname} ({local.stat().st_size/1e9:.1f} GB) in {time.time()-t1:.0f}s')
print(f'Total {time.time()-t0:.0f}s')

## 3. Open merged dataset + time subset

`dfm_tools.open_partitioned_dataset` merges the 4 FM partitions using UGRID conventions, removes ghost cells, and returns a single xugrid dataset.

Time selection: keep the pulse day (day 3) at high frequency, plus bracketing snapshots to see propagation + decay.

In [ ]:
import dfm_tools as dfmt
import xarray as xr
import numpy as np
import pandas as pd

pattern = str(SCRATCH / 'Stagnone_dxy01_15m_*_map.nc')
print(f'Opening: {pattern}')

# chunks={'time':1} per dfm_tools default — low memory, fast per-slice reads
uds = dfmt.open_partitioned_dataset(pattern)
print(uds)

In [ ]:
# Discover variable names (FM uses mesh2d_* prefix in map.nc)
print('Data vars in merged dataset:')
for v in sorted(uds.data_vars):
    print(f'  {v:40s}  dims={uds[v].dims}  shape={uds[v].shape}')
print()
print(f'Time range: {uds.time.values[0]} -> {uds.time.values[-1]}')
print(f'Time steps: {uds.sizes["time"]}')

In [ ]:
# Define time slices of interest.
# Day 3 pulse starts at 2025-07-03 00:00 (48h after sim start).
# mapInterval = 900 s = 15 min, so 4 slices per hour.
target_times = pd.to_datetime([
    # Pre-pulse baseline
    '2025-07-02 23:45',
    # Pulse window (2h at 15 min)
    '2025-07-03 00:15', '2025-07-03 00:45',
    '2025-07-03 01:15', '2025-07-03 01:45',
    # Post-pulse dispersion
    '2025-07-03 03:00', '2025-07-03 06:00',
    '2025-07-03 12:00', '2025-07-04 00:00',
    '2025-07-05 12:00', '2025-07-07 00:00',
    '2025-07-09 00:00', '2025-07-09 23:45',
    # Early hydro snapshots (confirm hypersaline init decay)
    '2025-07-01 00:00', '2025-07-01 03:00',
    '2025-07-01 06:00', '2025-07-01 12:00',
])
uds_t = uds.sel(time=target_times, method='nearest')
# Drop duplicates in case two targets round to the same existing slice
uds_t = uds_t.sel(time=~pd.DatetimeIndex(uds_t.time.values).duplicated())
print(f'Selected {uds_t.sizes["time"]} unique time slices.')
for t in uds_t.time.values:
    print(f'  {t}')

## 4. Drop non-essential variables

For v03c validation we need: water level, currents (u/v), salinity, temperature, 2 turbidity tracers, waves (even if constant).

In [ ]:
# Variable keep-list (be lenient — include any name containing these roots)
KEEP_ROOTS = [
    'waterlevel', 's1', 's0',              # water level
    'ucx', 'ucy', 'u1', 'v1',              # currents
    'salinity', 'sa1',                     # salinity
    'temperature', 'tem1',                 # temperature
    'turbid_airport', 'turbid_saltpans',   # our tracers
    'hwav', 'twav', 'phiwav', 'uorb',      # waves (constant, but check)
    'tausx', 'tausy',                      # wind/wave stress
    'windx', 'windy',                      # surface wind
    'rho',                                 # density (useful for plots)
]
keep = [v for v in uds_t.data_vars
        if any(root in v.lower() for root in KEEP_ROOTS)]
print(f'Keeping {len(keep)} variables:')
for v in keep:
    print(f'  {v}')

# Also keep mesh topology so xugrid can reconstruct the grid locally
uds_sub = uds_t[keep]
print()
print(f'Subset size estimate: {sum(uds_sub[v].nbytes for v in keep)/1e6:.1f} MB in-memory')

## 5. Save single merged subset NetCDF

In [ ]:
# xugrid.UgridDataset has to_netcdf; fallback to xr if needed
print(f'Writing {SUBSET_FILE} ...')
t1 = time.time()
try:
    uds_sub.ugrid.to_netcdf(str(SUBSET_FILE))
except AttributeError:
    # Fallback for older xugrid
    uds_sub.to_netcdf(str(SUBSET_FILE), engine='netcdf4')
print(f'  Written in {time.time()-t1:.0f}s, {SUBSET_FILE.stat().st_size/1e6:.1f} MB')

## 6. Upload subset to bucket for later download

In [ ]:
t1 = time.time()
r = subprocess.run(['mc', 'cp', str(SUBSET_FILE),
                    f'{S3_DST}{SUBSET_FILE.name}'],
                   capture_output=True, text=True)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError('mc cp failed')
print(f'Uploaded in {time.time()-t1:.0f}s')
print(f'Available at: s3://oidc-cmartinsjr/DFM_OUTPUT_SUBSET/{SUBSET_FILE.name}')
print()
print('Next step (locally):')
print('  python scripts/edito_sync.py download-subset --model-dir model/dflowfm_v03c')

## 7. (Optional) Clean scratch once subset is verified

The 56 GB of original map.nc copies can be freed on EDITO once the subset is saved.
Leave them for now if you want to re-run with different time selections.

In [ ]:
# Uncomment when ready to free EDITO scratch:
# for f in SCRATCH.glob('Stagnone_dxy01_15m_*_map.nc'):
#     f.unlink()
#     print(f'Removed {f.name}')
pass